In [0]:
"""
08_serial_number_events.py

Creates the Silver Serial Number Events table.

Input:
    parsed_events

Output:
    serial_number_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import col


# ============================================================
# Serial Number Events
# ============================================================

@dp.table(
    name="serial_number_events",
    comment="Validated manufacturing serial number assignment events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_execution_id",
    "execution_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_work_order_id",
    "work_order_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_serial_number",
    "serial_number IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_product_code",
    "product_code IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_status",
    "status IS NOT NULL",
)

def serial_number_events():

    df = dp.read_stream("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only serial number assignment events
        # -----------------------------------------

        .filter(
            col("event_type") == "SERIAL_NUMBER_ASSIGNED"
        )

        # -----------------------------------------
        # Business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "execution_id",
            "work_order_id",

            "serial_number",

            "product_code",

            "source_system",
            "correlation_id",

            "bronze_ingestion_timestamp",
            "silver_processing_timestamp",

            "payload.product_name",

            "payload.family",

            "payload.status",

        )

    )